<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/sample-answers/Bayesian_Inference_Assignment_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Sample Answer

### The 2PL Item Response Probability Model

Assuming that the item responses are conditionally independent given $\Theta=\theta$, the likelihood is

$$
P(Y=y\mid \Theta=\theta)=\prod_{i=1}^n
\left[
p_i(\theta)
\right]^{y_i}
\left[
1-p_i(\theta)
\right]^{1-y_i},
$$

where

$$
p_i(\theta)
=\frac{1}{1+e^{-a_i(\theta-b_i)}}.
$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define the 2PL Item Response Function
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate a range of latent ability values (theta)
theta_vals = np.linspace(-6, 6, 300)

# Define configurations to plot
# Two distinct a_i values (0.5 and 1.5)
# For a_i = 1.5, we include 3 different b_i values (-2, 0, 2)
curves = [
    {"a": 0.5, "b": 0, "line_style": "dash"},
    {"a": 1.5, "b": -2, "line_style": "solid"},
    {"a": 1.5, "b": 0, "line_style": "solid"},
    {"a": 1.5, "b": 2, "line_style": "solid"},
]

# Create the Plotly figure
fig = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]
    style = curve["line_style"]

    # Calculate probabilities
    p_vals = p_i(theta_vals, a, b)

    # Add trace to the plot
    fig.add_trace(go.Scatter(
        x=theta_vals,
        y=p_vals,
        mode='lines',
        name=f"a = {a}, b = {b}",
        line=dict(dash=style, width=2.5)
    ))

# Customize layout
fig.update_layout(
    title={
        'text': "Two-Parameter Logistic (2PL) Item Response Curves",
        'y':0.9,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor='rgba(0,0,0,0.1)'),
    yaxis=dict(range=[0, 1.05], gridcolor='rgba(0,0,0,0.1)'),
    template="plotly_white",
    legend=dict(
        yanchor="top",
        y=0.95,
        xanchor="left",
        x=0.05,
        bgcolor="rgba(255,255,255,0.8)"
    )
)

# Display the interactive plot
fig.show()

### Posterior Distribution

At step $k$ (for $k = 1, \dots, n$), the current item response is $y_k$, and its likelihood contribution is:

$$L(y_k \mid \theta) = p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k}$$

The sequential posterior density of $\Theta$ given the running response history vector $\mathbf{y}^{(k)}$ is updated recursively as follows:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) = \frac{ \left[ p_k(\theta)^{y_k} (1 - p_k(\theta))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) }{ \int_{\mathbb R} \left[ p_k(s)^{y_k} (1 - p_k(s))^{1 - y_k} \right] f_{\Theta\mid\mathbf{Y}^{(k-1)}}(s\mid\mathbf{y}^{(k-1)})\,ds }$$

**Definition of Terms:**

* $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ is the **prior density** for the current step (which is the posterior inherited from the previous step).
* For the very first item ($k=1$), the base prior is the initialized distribution:

$$f_{\Theta\mid\mathbf{Y}^{(0)}}(\theta\mid\mathbf{y}^{(0)}) = f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}}\exp\left(-\frac{\theta^2}{2}\right)$$


* The denominator is the **local normalizing constant**, integrating out $\theta$ across the current likelihood-prior product to ensure the area under the new step-$k$ density curve integrates to 1.

### Baye's Estimate and the MAP Estimate

Under a sequential framework, point estimates like the **Posterior Mean** (Bayes estimate under squared-error loss) or the **Maximum A Posteriori (MAP)** estimate are updated dynamically at each step $k$ using the running posterior density $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

1. Running Posterior Mean (Bayes Estimate)
The Bayesian estimate of the user's latent ability at step $k$, minimizing the expected squared-error loss, is the expected value of the current posterior distribution:
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \mathbb E[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \int_{\mathbb R} \theta \, f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})\,d\theta$$

2. Running Maximum A Posteriori (MAP) Estimate
Alternatively, the most likely ability parameter at step $k$ corresponds to the mode (peak) of the current posterior density curve:
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} \in \arg\max_{\theta\in\mathbb R} f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$$

**Computation Note**

In practice, as the user answers more items, the computational grid tracks the updated array values for $f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})$.

* $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ is updated via a running vector dot-product: `np.trapz(theta * current_posterior, theta)`.
* $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ is extracted simply via the index of the maximum value: `theta[np.argmax(current_posterior)]`.

### Numerical Implementation

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# =====================================================================
# PART 1: SEQUENTIAL BAYESIAN UPDATE (MANUAL 4-ITEM SIMULATION)
# =====================================================================

# 1. Define a fine grid of latent ability values (theta)
theta = np.linspace(-5, 5, 500)

# 2. Initialize the prior distribution: Standard Normal N(0, 1)
prior = stats.norm.pdf(theta, 0, 1)

# 3. Define the Item Response Function (2PL Model)
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 4. Define the manual simulated sequence of item encounters
running_items = [
    {"a": 1.0, "b": -1.5, "y": 1},  # Step 1: Got an easy item correct
    {"a": 1.5, "b": 0.5,  "y": 1},  # Step 2: Got a medium-hard item correct
    {"a": 1.2, "b": 1.5,  "y": 0},  # Step 3: Got a very hard item incorrect
    {"a": 2.0, "b": 0.2,  "y": 1}   # Step 4: Got a highly discriminative item correct
]

# Create the first Plotly figure
fig1 = go.Figure()

# Add the initial Prior distribution trace
fig1.add_trace(go.Scatter(
    x=theta,
    y=prior,
    mode='lines',
    name='Initial Prior: N(0,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# Run the sequential update loop
current_posterior = prior.copy()

for idx, item in enumerate(running_items):
    a = item["a"]
    b = item["b"]
    y = item["y"]

    # Calculate item response probability across the grid
    prob = p_i(theta, a, b)

    # Calculate the likelihood contribution of this response
    likelihood = (prob ** y) * ((1 - prob) ** (1 - y))

    # Posterior proportional to Prior * Likelihood
    current_posterior = current_posterior * likelihood

    # Numerically normalize the density curve using the trapezoidal rule
    integral = np.trapezoid(current_posterior, theta)
    current_posterior /= integral

    # Format trace labels
    result_text = "Correct" if y == 1 else "Incorrect"
    trace_name = f"Step {idx+1}: After Item {idx+1} ({result_text}, a={a}, b={b})"

    # Add the current posterior step to the plot
    fig1.add_trace(go.Scatter(
        x=theta,
        y=current_posterior,
        mode='lines',
        name=trace_name,
        line=dict(width=2)
    ))

# Customize layout for Figure 1
fig1.update_layout(
    title={
        'text': "Sequential Bayesian Update of User Ability (θ)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Latent Ability Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="left", x=0.02,
        bgcolor="rgba(255,255,255,0.7)"
    )
)

# Display the first plot
fig1.show()


# =====================================================================
# PART 2: PERFORMANCE TRACKING & CONVERGENCE TIMELINE (20 ITEMS)
# =====================================================================

# Set random seed for reproducibility of the random items/responses
np.random.seed(42)

# 1. Setup Simulation Parameters
theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)  # Finer grid for tracking accuracy

# 2. Generate Random Item Characteristics
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

# 3. Initialize Tracking Arrays (Step 0 = Initial Prior State)
running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

# Initialize prior density distribution: N(0, 1)
current_posterior_sim = stats.norm.pdf(theta_grid, 0, 1)

# 4. Run the 20-item Sequential Simulation loop
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Compute true response probability at true theta
    prob_true = p_i(theta_true, a_k, b_k)

    # Simulate stochastically generated user response y_k
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Calculate likelihood curve across grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))

    # Update and normalize
    current_posterior_sim = current_posterior_sim * likelihood
    integral_sim = np.trapezoid(current_posterior_sim, theta_grid)
    current_posterior_sim /= integral_sim

    # Compute point estimates
    theta_bayes_k = np.trapezoid(theta_grid * current_posterior_sim, theta_grid)
    theta_map_k = theta_grid[np.argmax(current_posterior_sim)]

    # Store estimates
    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

# 5. Create the second Plotly figure
fig2 = go.Figure()

# Add True Ability Reference Horizontal Line
fig2.add_hline(
    y=theta_true,
    line_dash="dash",
    line_color="red",
    line_width=2,
    annotation_text=f"True Ability (θ = {theta_true})",
    annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes,
    mode='lines+markers',
    name='Posterior Mean (θ̂_Bayes)',
    line=dict(color='blue', width=2.5),
    marker=dict(size=6)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map,
    mode='lines+markers',
    name='MAP Estimate (θ̂_MAP)',
    line=dict(color='green', width=2),
    marker=dict(size=6, symbol='square')
))

# Customize Layout for Figure 2
fig2.update_layout(
    title={
        'text': "Convergence of Latent Ability Estimators (θ) Over Time",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Sequence / Item Position (k)",
    yaxis_title="Estimated Ability (θ̂)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)

# Display the second plot
fig2.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Structural Probability and Properties**

Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

#### **2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and \beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$.

#### **4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

#### **5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

In [7]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Set random seed for reproducibility
np.random.seed(42)

# =====================================================================
# CONFIGURATION & PARAMETERS
# =====================================================================
theta_true = 0.35  # The true click-through rate of the advertisement (35%)
n_impressions = 100
steps = list(range(n_impressions + 1))

# Initialize Beta Prior Parameters (Prior: Uniform distribution Beta(1,1))
alpha_param = 1
beta_param = 1

# Setup grid for plotting probability density curves (Domain: 0 to 1)
theta_grid = np.linspace(0, 1, 500)

# Define specific milestone steps where we want to capture the full curve shape
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

# Tracking arrays for point estimates
running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]  # Mode of Uniform(0,1)

# Initialize Figure 1 for the probability density progression curves
fig1 = go.Figure()

# Plot the initial Prior Beta(1, 1) distribution curve
prior_density = stats.beta.pdf(theta_grid, alpha_param, beta_param)
fig1.add_trace(go.Scatter(
    x=theta_grid, y=prior_density, mode='lines',
    name='Initial Prior: Beta(1,1)',
    line=dict(dash='dash', width=2.5, color='gray')
))

# =====================================================================
# RUNNING ANALYTICAL UPDATE LOOP
# =====================================================================
for k in range(1, n_impressions + 1):
    # Simulate a user action stochastically based on true CTR
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # EXACT CLOSED-FORM CONJUGATE UPDATE RULES:
    alpha_param += y_k
    beta_param += (1 - y_k)

    # Calculate exact point estimates directly from analytical formulas
    theta_bayes_k = alpha_param / (alpha_param + beta_param)

    # Guard formula for mode when alpha or beta <= 1
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    # If the current step hits one of our milestone checkpoints, capture its PDF
    if k in milestones:
        # Evaluate exact density curve across the grid analytically
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No Click"

        fig1.add_trace(go.Scatter(
            x=theta_grid, y=density_k, mode='lines',
            name=f"Step {k}: After Event ({result_text}, α={alpha_param}, β={beta_param})",
            line=dict(width=2)
        ))

# =====================================================================
# VISUALIZE FIGURE 1: POSTERIOR DISTRIBUTION PROGRESSION
# =====================================================================
# Add True CTR vertical line to see where the density curves are peaking
fig1.add_vline(
    x=theta_true, line_dash="dot", line_color="red", line_width=2,
    annotation_text=f"True CTR ({theta_true})", annotation_position="top right"
)

fig1.update_layout(
    title={
        'text': "Analytical Posterior Density Progression (Beta-Binomial Updates)",
        'y': 0.95, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Conversion Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ | y)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=0.95, xanchor="right", x=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    )
)
fig1.show()

# =====================================================================
# VISUALIZE FIGURE 2: CONVERGENCE TIMELINE
# =====================================================================
fig2 = go.Figure()

# Add True CTR Reference Line
fig2.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right"
)

# Add Posterior Mean Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_bayes, mode='lines',
    name='Exact Posterior Mean (Beta Formula)',
    line=dict(color='blue', width=2.5)
))

# Add MAP Trace
fig2.add_trace(go.Scatter(
    x=steps, y=running_map, mode='lines',
    name='Exact MAP Estimate (Beta Formula)',
    line=dict(color='green', width=1.5, dash='dot')
))

fig2.update_layout(
    title={
        'text': "Analytical Beta-Binomial Conjugate Update Timeline",
        'y': 0.93, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'
    },
    xaxis_title="Number of User Impressions (k)",
    yaxis_title="Estimated Conversion Rate (θ̂)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()